In [4]:
import pandas as pd
import nltk
from collections import Counter
import numpy as np
nltk.download("punkt")
from sklearn.metrics import precision_score, recall_score, f1_score

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/samarthmahendra/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Notebook 3: Multilayer Perceptron
===============

CS 6120 Natural Language Processing, Amir



#### Marasanige Samarth Mahendra

Saving notebooks as pdfs
----------

Feel free to add cells to this notebook as you wish. Make sure to leave **code that you've written** and any **answers to questions** that you've written in your notebook. Turn in your notebook as a pdf at the end of lecture's day.


To convert your notebook to a pdf for turn in, you'll do the following:
1. Kernel -> Restart & Run All (clear your kernel's memory and run all cells)
2. File -> Download As -> .html -> open in a browser -> print to pdf

(The download as pdf option doesn't preserve formatting and output as nicely as taking the step "through" html, but will do if the above doesn't work for you.)

Task 1: Implement a Multilayer Perceptron for text classification in Torch
-------

In this notebook you will get to implement neural text classifiers using [Torch](https://pytorch.org/), a very popular deep learning framework. You may need to consult the documentation but since you will need to use this framework for the upcoming homework assignments this is an opportunity to get familiarized with it. 

The goal is to build neural binary classifiers to predict the toxicity (i.e., toxic vs non-toxic) of a post using data from the [Jigsaw Unintended Bias in Toxicity Classification competition](https://www.kaggle.com/c/jigsaw-unintended-bias-in-toxicity-classification) (here we will use a very small subset of the data). 

Recall that a MLP with $l$ hidden layers makes predictions as

$z = g(V^{l}\ldots g(V^2g(V^1f(x))))\\$
$P(\hat{y}|x) = \text{softmax}(Wz)$ 

where $f(x)$ is a feature representation of the input and hidden layer $j$ produces a new feature vector via a linear transformation paramterized by a weight vector $V^j$ followed by an activation function $g(\cdot)$ (i.e., an elementwise non-linear transformation). We recommend implementing your network using the [nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) module but you dont have to. 

The provided code is based on the feedforward neural network for the XOR problem that we saw in class. We also provide code to read the data and build BOW feature vectors. By default the code subsamples the training/test data to make development faster. Feel free to play with the full dataset if time permits.

In [8]:
import torch

if torch.backends.mps.is_available():
    print("MPS device is available.")
else:
    print("MPS device is not available.")

MPS device is available.


In [26]:
#NEURAL NETWORK DEFINITION

import torch
import torch.nn as nn
from torch import optim
import numpy as np
import random
import torch.nn.init as init
# fix the randomness to ensure reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
random.seed(42)

class MLP(nn.Module):
    """
    Defines the core neural network for doing multiclass classification over a single datapoint at a time. The network can be instantiated with arbitrary architectures (by which we mean number and size of hidden layers) 
    e.g., architecture = [1000, 50, 2] is a MLP with input layer of size 1000, hidden layer of size 50 and output layer of size 2
    
    Recall that the hidden layer is computed as a linear transformation followed by an activation function (i.e., non-linearity)
    Linear transformations are implemented with the nn.Linear() 
    Note 1: the input layer should have the same size as the input feature vectors and the output layer should be the number of classes.
    Note 2: be sure to match the input and output dimensions of all the layers
    """
    def __init__(self, architecture, init_method='xavier', activation='relu'):
        """
        Constructs the computation graph by instantiating the various layers and initializing weights.
        :param architecture: dimensions of all the layers (list)        
        """
        super(MLP, self).__init__()
        
        activation_functions = {
            'relu': nn.ReLU(),
            'sigmoid': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }
        
        # Create the layers based on the architecture provided
        layers = []
        for i in range(len(architecture) - 1):
            linear_layer = nn.Linear(architecture[i], architecture[i+1])
            
            # Initialize weights based on the specified method - Xavier or Glorot initialization
            if init_method == 'xavier':
                init.xavier_uniform_(linear_layer.weight)
            elif init_method == 'zero':
                init.zeros_(linear_layer.weight)
            
            layers.append(linear_layer)
            
            if i < len(architecture) - 2:  # Add activation function between layers except the last layer
                layers.append(activation_functions[activation])
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        """
        Runs the neural network on the given data and returns log probabilities of the various classes.

        :param x: a [inp]-sized tensor of input data
        :return: an [out]-sized tensor of log probabilities. (In general your network can be set up to return either log
        probabilities or a tuple of (loss, log probability) if you want to pass in y to this function as well
        """
        # Pass the input through the layers
        output = self.model(x)
        return torch.log_softmax(output, dim=-1)  # Return log probabilities

    def predict(self, x):
        with torch.no_grad():
            log_probs = self.forward(x)
            return torch.argmax(log_probs, dim=-1)
    
def form_input(x) -> torch.Tensor:
    """
    Form the input to the neural network. In general this may be a complex function that synthesizes multiple pieces
    of data, does some computation, handles batching, etc.

    :param x: a [num_samples x inp] numpy array containing input data
    :return: a [num_samples x inp] Tensor
    """
    return torch.from_numpy(x).float()

def train_model(model, train_xs, train_ys, num_classes, num_epochs, learning_rate):

    optimizer = optim.SGD(model.parameters(), lr=learning_rate)
    for epoch in range(0, num_epochs):
        ex_indices = [i for i in range(0, len(train_xs))]
        random.shuffle(ex_indices)
        total_loss = 0.0
        for idx in ex_indices:
            x = form_input(train_xs[idx])
            y = train_ys[idx]
            # Build one-hot representation of y. Instead of the label 0 or 1, y_onehot is either [0, 1] or [1, 0]. This
            # way we can take the dot product directly with a probability vector to get class probabilities.
            y_onehot = torch.zeros(num_classes)
            # scatter will write the value of 1 into the position of y_onehot given by y
            y_onehot.scatter_(0, torch.from_numpy(np.asarray(y,dtype=np.int64)), 1)
            # Zero out the gradients from the model object. *THIS IS VERY IMPORTANT TO DO BEFORE CALLING BACKWARD()*
            model.zero_grad()        
            log_probs = model.forward(x)
            # Can also use built-in NLLLoss as a shortcut but we're being explicit here
            loss = torch.neg(log_probs).dot(y_onehot)
            total_loss += loss
            # Computes the gradient and takes the optimizer step
            loss.backward()
            optimizer.step()
        print("Total loss on epoch %i: %f" % (epoch, total_loss))
    return model

In [27]:
#FRAMEWORK CODE 

def read_data(path, sample_frac=1):
    df = pd.read_csv(path)
    df = df.iloc[:int(len(df)*sample_frac)]
    y = df["label"]
    x = df["comment_text"]
    return x, np.array(y).astype(np.float32)

def build_vocab(X):
    MIN_FREQ = 3
    ct = Counter()
    for x_i in X:
        ct.update(nltk.word_tokenize(x_i.lower().strip()))
    #only keep words longer than 2 characters that occur at least MIN_FREQ times
    vocab = {k:i for i,k in enumerate([k for (k,v) in ct.most_common() if v > MIN_FREQ and len(k)>1])}
    return vocab

def build_BOW(X, vocab):
    bows = []
    for x_i in X:
        bow = np.zeros(len(vocab)).astype(np.float32)
        tokens = nltk.word_tokenize(x_i.lower().strip())
        for t in tokens:
            if t in vocab:
                bow[vocab[t]]+=1
        bows.append(bow)
    return np.array(bows)

def report_metrics(classifier, test_data, golds):
  """
    Applies the trained classifier to test data and computes performance
  """
#   golds = [data[1] for data in test_data]
  classified = [classifier.predict(form_input(data)) for data in test_data]
  print("Precision:", precision_score(golds, classified))
  print("Recall:", recall_score(golds, classified))
  print("F1:", f1_score(golds, classified))

In [30]:
#read train/test data
train_docs, train_ys = read_data("toxicity_small_train.csv",sample_frac=0.5)
test_x, test_y = read_data("toxicity_small_test.csv",sample_frac=0.5)
#build vocabulary
vocab = build_vocab(train_docs)
#extract bag-of-word features
train_xs = build_BOW(train_docs, vocab)
test_xs = build_BOW(test_x, vocab)

#Network definition
input_layer_d = train_xs.shape[1]
hidden_layer_d = 200
num_classes = 2
architecture = [input_layer_d, hidden_layer_d, num_classes]
#this just an example of how to instantiate the model 
model = MLP([input_layer_d, hidden_layer_d, num_classes])

# RUN TRAINING AND TEST
num_epochs = 10
initial_learning_rate = 0.01
model = train_model(model, train_xs, train_ys, num_classes, num_epochs, initial_learning_rate)

print(" === Train Set Performance === ")
report_metrics(model, train_xs, train_ys)

print(" === Test Set Performance === ")
report_metrics(model, test_xs, test_y)


# Define a function to train and compare models
def compare_initializations(train_xs, train_ys, test_xs, test_ys, architecture, num_classes, num_epochs, learning_rate):
    print("\n===== Training with Xavier Initialization =====")
    xavier_model = MLP(architecture, init_method='xavier')
    xavier_model = train_model(xavier_model, train_xs, train_ys, num_classes, num_epochs, learning_rate)
    
    print("\n=== Train Set Performance (Xavier) ===")
    report_metrics(xavier_model, train_xs, train_ys)
    
    print("\n=== Test Set Performance (Xavier) ===")
    report_metrics(xavier_model, test_xs, test_ys)
    
    print("\n===== Training with Zero Initialization =====")
    zero_model = MLP(architecture, init_method='zero')
    zero_model = train_model(zero_model, train_xs, train_ys, num_classes, num_epochs, learning_rate)
    
    print("\n=== Train Set Performance (Zero Init) ===")
    report_metrics(zero_model, train_xs, train_ys)
    
    print("\n=== Test Set Performance (Zero Init) ===")
    report_metrics(zero_model, test_xs, test_ys)
    
    
    # Define a function to train and compare models
def compare_initializations_with_different_activations(train_xs, train_ys, test_xs, test_ys, architecture, num_classes, num_epochs, learning_rate):
    print("\n===== Training with Xavier Initialization ===== ")
    xavier_model = MLP(architecture, init_method='xavier', activation='relu')
    xavier_model = train_model(xavier_model, train_xs, train_ys, num_classes, num_epochs, learning_rate)
    
    print("\n=== Train Set Performance (Xavier) Relu ===")
    report_metrics(xavier_model, train_xs, train_ys)
    
    print("\n=== Test Set Performance (Xavier) Relu  ===")
    report_metrics(xavier_model, test_xs, test_ys)
   
    print("\n===== Training with Xavier Initialization ===== ")
    xavier_model = MLP(architecture, init_method='xavier', activation='sigmoid')
    xavier_model = train_model(xavier_model, train_xs, train_ys, num_classes, num_epochs, learning_rate)
    
    print("\n=== Train Set Performance (Xavier) Sigmoid ===")
    report_metrics(xavier_model, train_xs, train_ys)
    
    print("\n=== Test Set Performance (Xavier) Sigmoid  ===")
    report_metrics(xavier_model, test_xs, test_ys)
    
    print("\n===== Training with Xavier Initialization ===== ")
    xavier_model = MLP(architecture, init_method='xavier', activation='tanh')
    xavier_model = train_model(xavier_model, train_xs, train_ys, num_classes, num_epochs, learning_rate)
    
    print("\n=== Train Set Performance (Xavier) Tanh ===")
    report_metrics(xavier_model, train_xs, train_ys)
    
    print("\n=== Test Set Performance (Xavier) Tanh  ===")
    report_metrics(xavier_model, test_xs, test_ys)
    
    
    



Total loss on epoch 0: 2307.815430
Total loss on epoch 1: 1911.533936
Total loss on epoch 2: 1535.469116
Total loss on epoch 3: 1210.802246
Total loss on epoch 4: 913.591980
Total loss on epoch 5: 676.718140
Total loss on epoch 6: 374.110107
Total loss on epoch 7: 288.716461
Total loss on epoch 8: 277.482666
Total loss on epoch 9: 216.215775
Total loss on epoch 10: 424.319580
Total loss on epoch 11: 219.726028
Total loss on epoch 12: 89.113846
Total loss on epoch 13: 61.442032
Total loss on epoch 14: 51.380169
Total loss on epoch 15: 45.682232
Total loss on epoch 16: 40.646782
Total loss on epoch 17: 37.576302
Total loss on epoch 18: 33.862061
Total loss on epoch 19: 33.653351
 === Train Set Performance === 
Precision: 1.0
Recall: 0.9965075669383003
F1: 0.9982507288629737
 === Test Set Performance === 
Precision: 0.8135095447870778
Recall: 0.7194805194805195
F1: 0.7636113025499656


#### Q1: Experiment with a couple of different MLP architectures. What do you observe?

1 Smaller Architecture: Fewer Layers and Units
Observation:
The model trains faster since there are fewer parameters to optimize.
However, with fewer hidden units or layers, the model may struggle to capture complex patterns in the data, leading to a higher loss and lower accuracy.
The model can underfit the data if the architecture is too simple, meaning it doesn't have enough capacity to learn the underlying structure of the dataset.
2 Larger Architecture: More Layers and Units
The model can capture more complex patterns due to the increased number of parameters (weights and biases).
Training time increases significantly because there are more parameters to optimize, which requires more computation.
While the model is more expressive and can potentially achieve better accuracy, there is a risk of overfitting, especially if the dataset is small.
Overfitting can be observed if the training loss decreases but the validation/test loss increases after some epochs.

#### Q2: Compare the performance your MLPs with Logistic Regression (note that this is just a MLP *without* any hidden layers). Are the results what you expected to see?

On problems that require more expressive models (non-linearly separable data), Logistic Regression tends to underfit. It struggles to learn the intricate relationships in the data, resulting in lower accuracy and higher loss compared to an MLP with hidden layers.

In [32]:
compare_initializations(train_xs, train_ys, test_xs, test_y, architecture, num_classes, num_epochs, initial_learning_rate)



===== Training with Xavier Initialization =====
Total loss on epoch 0: 2296.122314
Total loss on epoch 1: 1907.511475
Total loss on epoch 2: 1563.022583
Total loss on epoch 3: 1181.557983
Total loss on epoch 4: 915.492676
Total loss on epoch 5: 645.993591
Total loss on epoch 6: 367.960449
Total loss on epoch 7: 257.657623
Total loss on epoch 8: 148.647186
Total loss on epoch 9: 116.735001
Total loss on epoch 10: 98.320915
Total loss on epoch 11: 71.326080
Total loss on epoch 12: 60.779385
Total loss on epoch 13: 54.052677
Total loss on epoch 14: 48.177212
Total loss on epoch 15: 43.699799
Total loss on epoch 16: 39.750816
Total loss on epoch 17: 37.732117
Total loss on epoch 18: 35.609959
Total loss on epoch 19: 32.943649

=== Train Set Performance (Xavier) ===
Precision: 1.0
Recall: 0.9965075669383003
F1: 0.9982507288629737

=== Test Set Performance (Xavier) ===
Precision: 0.803030303030303
Recall: 0.6883116883116883
F1: 0.7412587412587412

===== Training with Zero Initialization ===

#### Q3: Investigate the impact of [initialization](https://pytorch.org/docs/stable/nn.init.html) of weight matrices using your best performing MLP. Compare Xavier and Glorot initialization with zero initialization. 

Zero Initialization:
On the train set, the model performs well with decent precision and recall. This suggests the model can learn to some extent, but the initialization causes problems when generalizing to the test set.


Xavier Initialization:
On the train set, the performance is near perfect, showing that the model learned well from the training data with no major issues.
On the test set, there is a general improvement in precision compared to Zero Initialization, but the recall is lower, which suggests that while the model predicts positives more accurately, it misses more actual positives.

In [33]:
compare_initializations_with_different_activations(train_xs, train_ys, test_xs, test_y, architecture, num_classes, num_epochs, initial_learning_rate)



===== Training with Xavier Initialization ===== 
Total loss on epoch 0: 2294.603271
Total loss on epoch 1: 1931.668701
Total loss on epoch 2: 1529.914062
Total loss on epoch 3: 1265.432129
Total loss on epoch 4: 953.914368
Total loss on epoch 5: 600.596985
Total loss on epoch 6: 411.430847
Total loss on epoch 7: 265.456573
Total loss on epoch 8: 154.664627
Total loss on epoch 9: 109.314545
Total loss on epoch 10: 83.868706
Total loss on epoch 11: 68.659485
Total loss on epoch 12: 60.560486
Total loss on epoch 13: 52.165157
Total loss on epoch 14: 46.615520
Total loss on epoch 15: 42.989483
Total loss on epoch 16: 38.515560
Total loss on epoch 17: 37.679779
Total loss on epoch 18: 35.316933
Total loss on epoch 19: 32.654102

=== Train Set Performance (Xavier) Relu ===
Precision: 1.0
Recall: 0.9965075669383003
F1: 0.9982507288629737

=== Test Set Performance (Xavier) Relu  ===
Precision: 0.8066378066378066
Recall: 0.7259740259740259
F1: 0.7641831852358169

===== Training with Xavier Ini

#### Q4: Investigate the impact of [non-linear activations](https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity). See Torch documentation on the available non-linear functions and compare the performance of 3 different functions (e.g., sigmoid, relu and tanh).

ReLU: Overfits the training set and performs poorly on the test set, having excellent precision on the training set. Sigmoid performs worse than Tanh on both the training and testing sets, most likely due to vanishing gradient difficulties. Tanh strikes a better compromise between overfitting and generalization, making it an excellent choice in this case.